# Overview

This notebook runs the setup for the agent. It guides you through the following steps:

1. **Schema and Volume Setup:** Creates a dedicated schema and volume for your data.
2. **Permissions:** Grants necessary permissions to all account users for collaborative access.
3. **Data Upload:** Uploads sample CSV files to the Databricks volume.
4. **Delta Table Creation:** Reads the uploaded CSV files and creates Delta tables, including enabling Change Data Feed for the `supplier_sops` table.
5. **Vector Search Endpoint:** Initializes and waits for a Databricks Vector Search endpoint.
6. **Index Creation:** Not in this notebook. Index will be created in `/uc_tools` during tool setup
By the end of this workflow, all the data dependencies for the agent are setup

In [0]:
%pip install -U -qqqq unitycatalog-ai[databricks] mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents python-dotenv uv
%restart_python

In [0]:
dbutils.widgets.removeAll()

In [0]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())  

# Create widgets in order to bind environment variables to the SQL session variables so they can be used in %sql cells
dbutils.widgets.text("TARGET_CATALOG", os.getenv("TARGET_CATALOG"))
dbutils.widgets.text("TARGET_SCHEMA", os.getenv("TARGET_SCHEMA"))
dbutils.widgets.text("VS_INDEX_BASE_TABLE", os.getenv("VS_INDEX_BASE_TABLE"))
dbutils.widgets.text("VS_INDEX", os.getenv("VS_INDEX"))

TARGET_CATALOG = dbutils.widgets.get("TARGET_CATALOG")
TARGET_SCHEMA = dbutils.widgets.get("TARGET_SCHEMA")
VS_INDEX_BASE_TABLE = dbutils.widgets.get("VS_INDEX_BASE_TABLE")
VS_INDEX = dbutils.widgets.get("VS_INDEX")

print(f"TARGET_CATALOG: {TARGET_CATALOG}")
print(f"TARGET_SCHEMA: {TARGET_SCHEMA}")
print(f"VS_INDEX_BASE_TABLE: {VS_INDEX_BASE_TABLE}")
print(f"VS_INDEX: {VS_INDEX}")

In [0]:
%sql

-- Depending on your Unity Catalog setup, you may need to create manually, if not uncomment and rerun
-- CREATE CATALOG IF NOT EXISTS IDENTIFIER(:TARGET_CATALOG);

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:TARGET_CATALOG || "." || :TARGET_SCHEMA);
CREATE VOLUME IF NOT EXISTS IDENTIFIER(:TARGET_CATALOG || "." || :TARGET_SCHEMA || ".data");
USE IDENTIFIER(:TARGET_CATALOG || "." || :TARGET_SCHEMA);

In [0]:
spark.sql(
    f"GRANT USE SCHEMA, SELECT, EXECUTE, READ VOLUME ON SCHEMA {TARGET_CATALOG}.{TARGET_SCHEMA} TO `account users`"
)

## Synthetic data

The CSVs in this folder are generated by `generate_synthetic_data.py` (seeded, demo date 2026-11-18). Re-run it locally to regenerate:

```
python3 data/generate_synthetic_data.py
```

It writes `sites`, `facilities`, `inventory` and `shipments`. Weather is not stored: the agent's `check_weather` tool uses live forecasts. `suppliers.csv` and `supplier_sops.csv` are hand-maintained and are not changed by the script.


In [0]:
import os

DATA_PATH = os.getcwd() # current path of csv files
VOLUME_PATH = f"/Volumes/{os.environ['TARGET_CATALOG']}/{os.environ['TARGET_SCHEMA']}/data"

print(f"Removing existing CSVs from {VOLUME_PATH}...")
os.system(f"rm -f {VOLUME_PATH}/*.csv")

print(f"Copying CSVs from {DATA_PATH} to {VOLUME_PATH}...")
os.system(f"cp {DATA_PATH}/*.csv {VOLUME_PATH}")

In [0]:
%sql
-- shipments.csv -> table: shipments_base. Dates are stored as DAYS FROM THE DEMO ANCHOR (the
-- DEMO_DATE the CSVs were generated with), so the `shipments` view (next cell) can re-date them
-- against today on every read: the demo works on any day with no refresh job.
CREATE OR REPLACE TABLE shipments_base
SELECT * EXCEPT (_rescued_data, ship_date, eta_date, original_eta_date, procedure_date),
  datediff(CAST(ship_date AS DATE), DATE'2026-11-19')         AS ship_in_days,
  datediff(CAST(eta_date AS DATE), DATE'2026-11-19')          AS eta_in_days,
  datediff(CAST(original_eta_date AS DATE), DATE'2026-11-19') AS original_eta_in_days,
  datediff(CAST(procedure_date AS DATE), DATE'2026-11-19')    AS procedure_in_days
FROM read_files(
  "/Volumes/" || :TARGET_CATALOG || "/" || :TARGET_SCHEMA || "/data/shipments.csv",
  FORMAT => "csv",
  HEADER => true
);

-- suppliers.csv -> table: suppliers
CREATE OR REPLACE TABLE suppliers
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :TARGET_CATALOG || "/" || :TARGET_SCHEMA || "/data/suppliers.csv",
  FORMAT => "csv",
  HEADER => true
);

-- inventory.csv -> table: inventory_base (lot expiry stored as days from the demo anchor)
CREATE OR REPLACE TABLE inventory_base
SELECT * EXCEPT (_rescued_data, lot_expiry),
  datediff(CAST(lot_expiry AS DATE), DATE'2026-11-19') AS lot_expiry_in_days
FROM read_files(
  "/Volumes/" || :TARGET_CATALOG || "/" || :TARGET_SCHEMA || "/data/inventory.csv",
  FORMAT => "csv",
  HEADER => true
);

-- sites.csv -> table: sites  (Supply and distribution centers with lat/lon)
CREATE OR REPLACE TABLE sites
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :TARGET_CATALOG || "/" || :TARGET_SCHEMA || "/data/sites.csv",
  FORMAT => "csv",
  HEADER => true
);

-- facilities.csv -> table: facilities  (Hospitals and clinics with lat/lon and serving site)
CREATE OR REPLACE TABLE facilities
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :TARGET_CATALOG || "/" || :TARGET_SCHEMA || "/data/facilities.csv",
  FORMAT => "csv",
  HEADER => true
);

-- supplier_sops.csv -> table: supplier_sops  (multiline text with embedded quotes)
CREATE OR REPLACE TABLE IDENTIFIER(:VS_INDEX_BASE_TABLE)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :TARGET_CATALOG || "/" || :TARGET_SCHEMA || "/data/supplier_sops.csv",
  FORMAT => "csv",
  HEADER => true,
  MULTILINE => true,
  ESCAPE => '"'
);

In [ ]:
# Relative-date views: `shipments` and `inventory` re-date the base tables against TODAY
# (America/New_York) on every read, so agent, Genie, metric view, dashboard and UC functions
# always see current dates. DEMO_ANCHOR must match DEMO_DATE in generate_synthetic_data.py.
CAT, SCH = os.environ["TARGET_CATALOG"], os.environ["TARGET_SCHEMA"]
TODAY = "to_date(from_utc_timestamp(current_timestamp(), 'America/New_York'))"

def _drop_if_table(name):
    rows = spark.sql(
        f"SELECT table_type FROM {CAT}.information_schema.tables "
        f"WHERE table_schema = '{SCH}' AND table_name = '{name}'").collect()
    if rows and rows[0][0] != "VIEW":
        spark.sql(f"DROP TABLE {CAT}.{SCH}.{name}")

ship_cols = ["shipment_id", "product_id", "sku", "product_name", "supplier_id", "origin", "destination",
             f"date_add({TODAY}, ship_in_days) AS ship_date", f"date_add({TODAY}, eta_in_days) AS eta_date",
             "carrier", "status", "temperature_max_f", "temperature_min_f",
             f"date_add({TODAY}, original_eta_in_days) AS original_eta_date", "delay_hours", "delay_reason", "priority",
             f"date_add({TODAY}, procedure_in_days) AS procedure_date", "last_recorded_temp_f",
             "destination_facility_id", "destination_facility_name", "destination_facility_type", "routed_via_site_id",
             "origin_latitude", "origin_longitude", "destination_latitude", "destination_longitude", "route_miles"]
inv_cols = ["site_id", "site_name", "city", "state", "product_id", "sku", "product_name", "supplier_id",
            "on_hand_qty", f"date_add({TODAY}, lot_expiry_in_days) AS lot_expiry"]

for name, cols in (("shipments", ship_cols), ("inventory", inv_cols)):
    _drop_if_table(name)
    spark.sql(f"CREATE OR REPLACE VIEW {CAT}.{SCH}.{name} "
              f"COMMENT 'Dates are computed relative to today (America/New_York) from {name}_base.' AS "
              f"SELECT {', '.join(cols)} FROM {CAT}.{SCH}.{name}_base")
    n = spark.sql(f"SELECT count(*) FROM {CAT}.{SCH}.{name}").first()[0]
    print(f"{name}: view over {name}_base, {n} rows")
display(spark.sql(f"SELECT min(eta_date), max(eta_date), current_date() FROM {CAT}.{SCH}.shipments"))


In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

if TARGET_CATALOG not in [v["name"] for v in client.list_endpoints().get("endpoints", [])]:
    print(f"Creating endpoint {TARGET_CATALOG}")
    client.create_endpoint(name=TARGET_CATALOG, endpoint_type="STANDARD")

client.wait_for_endpoint(TARGET_CATALOG)
print(f"Created endpoint {TARGET_CATALOG}")